In [1]:
import os

root_dir = "medquad"  # path to your main folder

all_files = []
for root, dirs, files in os.walk(root_dir):
    for file in files:
        all_files.append(os.path.join(root, file))

print("Total files found:", len(all_files))
for f in all_files[:20]:
    print(f)

Total files found: 11090
medquad\medquad.csv
medquad\biomedical-ner-all\config.json
medquad\biomedical-ner-all\pytorch_model.bin
medquad\biomedical-ner-all\README.md
medquad\biomedical-ner-all\special_tokens_map.json
medquad\biomedical-ner-all\tokenizer.json
medquad\biomedical-ner-all\tokenizer_config.json
medquad\biomedical-ner-all\vocab.txt
medquad\BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext\config.json
medquad\BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext\pytorch_model.bin
medquad\BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext\special_tokens_map.json
medquad\BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext\tokenizer.json
medquad\BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext\tokenizer_config.json
medquad\BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext\vocab.txt
medquad\Bio_ClinicalBERT\config.json
medquad\Bio_ClinicalBERT\pytorch_model.bin
medquad\Bio_ClinicalBERT\special_tokens_map.json
medquad\Bio_ClinicalBERT\tokenizer.json
medquad\Bio_ClinicalBERT\

In [2]:
import os
from collections import Counter

root_dir = "medquad"

file_extensions = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        ext = os.path.splitext(file)[1].lower()  # get extension like .csv, .json
        if ext == "":
            ext = "NO_EXTENSION"
        file_extensions.append(ext)

# Count occurrences of each extension
ext_counts = Counter(file_extensions)

print("📂 File type inventory:")
for ext, count in ext_counts.most_common():
    print(f"{ext:15} : {count}")

📂 File type inventory:
.xml            : 10994
.json           : 35
NO_EXTENSION    : 19
.bin            : 13
.safetensors    : 8
.txt            : 6
.csv            : 5
.md             : 3
.png            : 3
.yaml           : 1
.nemo           : 1
.pt             : 1
.model          : 1


In [3]:
import os

xml_examples = []
for root, dirs, files in os.walk("medquad"):
    for f in files:
        if f.lower().endswith(".xml"):
            xml_examples.append(os.path.join(root, f))
        if len(xml_examples) >= 5:
            break
print("Example XML files:\n", "\n".join(xml_examples))

Example XML files:
 medquad\HC_DATA\medquad_xml\1_CancerGov_QA\0000001_1.xml
medquad\HC_DATA\medquad_xml\1_CancerGov_QA\0000001_2.xml
medquad\HC_DATA\medquad_xml\1_CancerGov_QA\0000001_3.xml
medquad\HC_DATA\medquad_xml\1_CancerGov_QA\0000001_4.xml
medquad\HC_DATA\medquad_xml\1_CancerGov_QA\0000001_5.xml
medquad\HC_DATA\medquad_xml\2_GARD_QA\0000004.xml
medquad\HC_DATA\medquad_xml\3_GHR_QA\0000001.xml
medquad\HC_DATA\medquad_xml\4_MPlus_Health_Topics_QA\0000001.xml
medquad\HC_DATA\medquad_xml\5_NIDDK_QA\0000001.xml
medquad\HC_DATA\medquad_xml\6_NINDS_QA\0000001.xml
medquad\HC_DATA\medquad_xml\7_SeniorHealth_QA\0000001.xml
medquad\HC_DATA\medquad_xml\8_NHLBI_QA_XML\0000001.xml
medquad\HC_DATA\medquad_xml\9_CDC_QA\0000001.xml
medquad\medquad_xml\1_CancerGov_QA\0000001_1.xml
medquad\medquad_xml\2_GARD_QA\0000004.xml
medquad\medquad_xml\3_GHR_QA\0000001.xml
medquad\medquad_xml\4_MPlus_Health_Topics_QA\0000001.xml
medquad\medquad_xml\5_NIDDK_QA\0000001.xml
medquad\medquad_xml\6_NINDS_QA\0000

In [4]:
# Focus on Neurology-Related Folders
neurology_dirs = [
    "6_NINDS_QA",   # National Institute of Neurological Disorders and Stroke
    "2_GARD_QA",    # Genetic and Rare Diseases (some neurological)
    "3_GHR_QA",     # Genetic Home Reference (some neurological)
]

In [5]:
# Parse Only Those XML Files
import xml.etree.ElementTree as ET
import pandas as pd
import os
from tqdm import tqdm

neurology_dirs = ["6_NINDS_QA", "2_GARD_QA", "3_GHR_QA"]

xml_data = []

for root, dirs, files in os.walk("medquad"):
    if any(ndir in root for ndir in neurology_dirs):
        for f in files:
            if f.lower().endswith(".xml"):
                file_path = os.path.join(root, f)
                try:
                    tree = ET.parse(file_path)
                    root_elem = tree.getroot()

                    # Try to extract metadata if available
                    disease = root_elem.findtext(".//disease")
                    for doc in root_elem.findall(".//document"):
                        q = doc.findtext(".//question")
                        a = doc.findtext(".//answer")
                        if q and a:
                            xml_data.append({
                                "focus_area": disease if disease else "Unknown",
                                "question": q.strip(),
                                "answer": a.strip(),
                                "source": f
                            })
                except Exception as e:
                    print(f"Error parsing {f}: {e}")

print(f"Extracted {len(xml_data)} question–answer pairs.")
df_xml = pd.DataFrame(xml_data)
os.makedirs("data", exist_ok=True)
df_xml.to_csv("data/medquad_neurology_extracted.csv", index=False)
print("Saved as data/medquad_neurology_extracted.csv")

Extracted 0 question–answer pairs.
Saved as data/medquad_neurology_extracted.csv


In [6]:
sample_path = r"medquad\HC_DATA\medquad_xml\6_NINDS_QA\0000001.xml"  # or any .xml inside 6_NINDS_QA
with open(sample_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 30:
            break
        print(line.strip())

<?xml version="1.0" encoding="UTF-8"?>
<Document id="0000001" source="NINDS" url="http://www.ninds.nih.gov/disorders/absence_septum_pellucidum/absence_septum_pellucidum.htm">
<Focus>Absence of the Septum Pellucidum</Focus>
<FocusAnnotations>
<UMLS>
<CUIs>
<CUI>C0431371</CUI>
</CUIs>
<SemanticTypes>
<SemanticType>T019</SemanticType>
</SemanticTypes>
<SemanticGroup>Disorders</SemanticGroup>
</UMLS>
</FocusAnnotations>
<QAPairs>
<QAPair pid="1">
<Question qid="0000001-1" qtype="information">What is (are) Absence of the Septum Pellucidum ?</Question>
<Answer>The septum pellucidum (SP) is a thin membrane located at the midline of the brain between the two cerebral hemispheres, or halves of the brain.. It is connected to the corpus callosum -- a collection of nerve fibers that connect the cerebral hemispherers. This rare abnormality accompanies various malformations of the brain that affect intelligence, behavior, and the neurodevelopmental process, and seizures may occur. Children who are b

In [7]:
# Corrected Parse
import xml.etree.ElementTree as ET
import pandas as pd
import os
from tqdm import tqdm

xml_data = []

for root, dirs, files in os.walk("medquad"):
    if "6_NINDS_QA" in root or "2_GARD_QA" in root or "3_GHR_QA" in root:
        for f in files:
            if f.lower().endswith(".xml"):
                file_path = os.path.join(root, f)
                try:
                    tree = ET.parse(file_path)
                    doc = tree.getroot()
                    
                    focus = doc.findtext("Focus")
                    source = doc.attrib.get("source", "Unknown")
                    url = doc.attrib.get("url", "Unknown")

                    # Extract Q&A pairs
                    for qa in doc.findall(".//QAPair"):
                        question = qa.findtext("Question")
                        answer = qa.findtext("Answer")
                        
                        if question and answer:
                            xml_data.append({
                                "focus_area": focus if focus else "Unknown",
                                "question": question.strip(),
                                "answer": answer.strip(),
                                "source": source,
                                "url": url
                            })
                except Exception as e:
                    print(f"Error parsing {f}: {e}")

print(f"Extracted {len(xml_data)} question–answer pairs.")
df_xml = pd.DataFrame(xml_data)
os.makedirs("data", exist_ok=True)
df_xml.to_csv("data/medquad_neurology_extracted.csv", index=False)
print("Saved as data/medquad_neurology_extracted.csv")


Extracted 23814 question–answer pairs.
Saved as data/medquad_neurology_extracted.csv


In [8]:
# Verify the Data
import pandas as pd

df = pd.read_csv("data/medquad_neurology_extracted.csv")
print("Total rows:", len(df))
print("\nColumns:", df.columns.tolist())
print("\nSample rows:")
print(df.sample(5))


Total rows: 23814

Columns: ['focus_area', 'question', 'answer', 'source', 'url']

Sample rows:
                                              focus_area  \
3255   Mitochondrial encephalomyopathy lactic acidosi...   
19708               Hutchinson-Gilford progeria syndrome   
680          Carbamoyl phosphate synthetase 1 deficiency   
14055               Hemangioma thrombocytopenia syndrome   
3702               Olivopontocerebellar atrophy deafness   

                                                question  \
3255   Is Mitochondrial encephalomyopathy lactic acid...   
19708  What are the genetic changes related to Hutchi...   
680    What is (are) Carbamoyl phosphate synthetase 1...   
14055  What is (are) Hemangioma thrombocytopenia synd...   
3702   What are the symptoms of Olivopontocerebellar ...   

                                                  answer source  \
3255   How is mitochondrial encephalomyopathy, lactic...   GARD   
19708  Mutations in the LMNA gene cause Hutchins

In [9]:
# Profile the Dataset (Top Focus Areas)
import pandas as pd

df = pd.read_csv("data/medquad_neurology_extracted.csv")

print("\nTop 25 Focus Areas:")
df['focus_area'].value_counts().head(25)


Top 25 Focus Areas:


focus_area
Danon disease                           22
Prader-Willi syndrome                   22
Poland syndrome                         22
Wolfram syndrome                        22
GM1 gangliosidosis                      22
Huntington disease                      22
Opitz G/BBB syndrome                    22
Greig cephalopolysyndactyly syndrome    22
Camurati-Engelmann disease              22
Laron syndrome                          22
Holt-Oram syndrome                      22
Langerhans cell histiocytosis           22
Peters plus syndrome                    22
Klinefelter syndrome                    22
Cornelia de Lange syndrome              22
Liddle syndrome                         22
21-hydroxylase deficiency               22
Cowden syndrome                         22
Bartter syndrome                        22
Ehlers-Danlos syndrome                  22
MECP2 duplication syndrome              22
Wilson disease                          20
Milroy disease                          20


In [10]:
# Identify Neurology-Relevant Topics
neuro_keywords = [
    "parkinson", "sclerosis", "epilepsy", "migraine", "alzheimer",
    "neuro", "brain", "nervous", "stroke", "spinal", "ataxia",
    "muscular", "dystrophy", "huntington", "neuropathy", "encephal",
    "seizure", "cerebral", "autism", "myopathy", "neuron", "neural"
]

neuro_df = df[df['focus_area'].str.contains('|'.join(neuro_keywords), case=False, na=False)]

print("Neurology-related pairs:", len(neuro_df))
print("\nTop 20 neurological conditions:\n", neuro_df['focus_area'].value_counts().head(20))

neuro_df.to_csv("data/neurology_chatbot_final.csv", index=False)

Neurology-related pairs: 3374

Top 20 neurological conditions:
 focus_area
Huntington disease                                22
Parkinson disease                                 20
Schizencephaly                                    20
Brody myopathy                                    20
Friedreich ataxia                                 18
Meesmann corneal dystrophy                        18
Wolff-Parkinson-White syndrome                    18
Leber hereditary optic neuropathy                 18
Anencephaly                                       16
Northern epilepsy                                 14
Laing distal myopathy                             14
Juvenile Huntington disease                       12
X-linked lissencephaly with abnormal genitalia    12
Neurofibromatosis                                 12
Oculopharyngeal muscular dystrophy                12
Spastic diplegia cerebral palsy                   12
Hydranencephaly                                   12
Best vitelliform macular

In [11]:
# Remove duplicates and very short answers
before = len(neuro_df)
neuro_df.drop_duplicates(subset=["question", "answer"], inplace=True)
neuro_df = neuro_df[neuro_df['answer'].str.len() > 30]
after = len(neuro_df)
print(f"Cleaned dataset: {after} pairs (removed {before - after} duplicates/short entries)")

# Save final version
neuro_df.to_csv("data/neurology_chatbot_clean.csv", index=False)

Cleaned dataset: 1687 pairs (removed 1687 duplicates/short entries)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_13140\2676527329.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  neuro_df.drop_duplicates(subset=["question", "answer"], inplace=True)


In [12]:
# Split into train, val, test
from sklearn.model_selection import train_test_split
import os

neuro_df = pd.read_csv("data/neurology_chatbot_clean.csv")

train, test = train_test_split(neuro_df, test_size=0.2, random_state=42)
val, test = train_test_split(test, test_size=0.5, random_state=42)

os.makedirs("data", exist_ok=True)
train.to_csv("data/train.csv", index=False)
val.to_csv("data/val.csv", index=False)
test.to_csv("data/test.csv", index=False)

print("Train:", len(train), "Val:", len(val), "Test:", len(test))

Train: 1349 Val: 169 Test: 169


In [13]:
# Neurology Chatbot Fine-Tuning Setup

import os
import math
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# Config

MODEL_NAME = "google/flan-t5-small"
DATA_DIR   = "data"
TRAIN_CSV  = os.path.join(DATA_DIR, "train.csv")
VAL_CSV    = os.path.join(DATA_DIR, "val.csv")
TEST_CSV   = os.path.join(DATA_DIR, "test.csv")
OUTDIR     = "checkpoints/flan_t5_small_neuro"

MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 192
BATCH_SIZE     = 8
EPOCHS         = 3
LR             = 3e-5
NUM_BEAMS      = 4
SEED           = 42

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs("logs", exist_ok=True)

print("Setup complete — ready for data loading.")

Setup complete — ready for data loading.


In [15]:
# Load and prepare neurology chatbot dataset (fixed)

train_df = pd.read_csv("data/train.csv")
val_df   = pd.read_csv("data/val.csv")
test_df  = pd.read_csv("data/test.csv")

# Rename columns if needed
rename_map = {"question": "user", "answer": "bot"}
train_df.rename(columns=rename_map, inplace=True)
val_df.rename(columns=rename_map, inplace=True)
test_df.rename(columns=rename_map, inplace=True)

print("Column names standardized.")
print("Train columns:", list(train_df.columns))

Column names standardized.
Train columns: ['focus_area', 'user', 'bot', 'source', 'url']


In [16]:
# Convert pandas → Hugging Face Datasets
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

print("\nData loaded and converted to Hugging Face Datasets.")
train_ds


Data loaded and converted to Hugging Face Datasets.


Dataset({
    features: ['focus_area', 'user', 'bot', 'source', 'url'],
    num_rows: 1349
})

In [17]:
# Tokenization & Preprocessing
from transformers import AutoTokenizer

MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 192

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer loaded.")

Tokenizer loaded.


In [18]:
# Tokenization function
def preprocess_batch(examples):
    # Encode user questions (inputs)
    model_inputs = tokenizer(
        examples["user"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )
    # Encode bot answers (targets)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["bot"],
            max_length=MAX_TARGET_LEN,
            truncation=True,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply to all splits
train_tok = train_ds.map(preprocess_batch, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(preprocess_batch,   batched=True, remove_columns=val_ds.column_names)

print("Tokenization complete.")
train_tok

Map:   0%|          | 0/1349 [00:00<?, ? examples/s]C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 169/169 [00:00<00:00, 3306.47 examples/s]

Tokenization complete.


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1349
})

In [19]:
# Model, Data Collator, and Trainer Setup
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, TrainingArguments

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("Model loaded successfully.")

# Data collator for dynamic padding
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training arguments (simplified for Transformers v4.57)
args = TrainingArguments(
    output_dir="checkpoints/flan_t5_small_neuro",
    eval_strategy="epoch",          # replaced evaluation_strategy
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    report_to=[],                   # disables wandb/tensorboard
    seed=42,
)

print("Training arguments ready.")

Model loaded successfully.
Training arguments ready.


In [20]:
# Trainer Setup & Fine-Tuning

from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
import numpy as np
from transformers import Trainer

# ---- Metrics function ----
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Handle model outputs (can be tuple)
    if isinstance(preds, tuple):
        preds = preds[0]

    # Convert tensors to NumPy
    preds = np.array(preds)
    labels = np.array(labels)

    # Sometimes preds come out as nested lists (logits instead of token IDs)
    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    # Replace masked tokens with pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Convert to integer lists
    preds = preds.tolist()
    labels = labels.tolist()

    # Decode
    pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BLEU
    bleu = corpus_bleu(pred_texts, [label_texts]).score

    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rougeL = np.mean([
        scorer.score(ref, hyp)['rougeL'].fmeasure
        for ref, hyp in zip(label_texts, pred_texts)
    ])

    return {"bleu": bleu, "rougeL": rougeL}

In [21]:
# ---- Trainer ----
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# reassign compute_metrics
trainer.compute_metrics = compute_metrics

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13140\3819880247.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [22]:
# ---- Train ----
# temporarily disable metrics
trainer.compute_metrics = None  

# run training
train_output = trainer.train()

print("\nTraining finished (metrics disabled during training)")
print(train_output)

C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,2.997800,2.456510
2,2.811500,2.300149
3,2.710800,2.259085


C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Training finished (metrics disabled during training)
TrainOutput(global_step=507, training_loss=2.848313420011682, metrics={'train_runtime': 3173.6274, 'train_samples_per_second': 1.275, 'train_steps_per_second': 0.16, 'total_flos': 38944423802880.0, 'train_loss': 2.848313420011682, 'epoch': 3.0})


In [23]:
# Evaluate once training is complete

# restore metrics function
trainer.compute_metrics = compute_metrics

# evaluate on the full validation (or test) set
metrics = trainer.evaluate(eval_dataset=val_tok)

print("\nPost-training evaluation results:")
for k, v in metrics.items():
    print(f"{k:15s}: {v:.4f}")

C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Post-training evaluation results:
eval_loss      : 2.2591
eval_bleu      : 19.3164
eval_rougeL    : 0.4064
eval_runtime   : 169.7058
eval_samples_per_second: 0.9960
eval_steps_per_second: 0.1300
epoch          : 3.0000


In [24]:
model.save_pretrained("checkpoints/flan_t5_neurology_v1")
tokenizer.save_pretrained("checkpoints/flan_t5_neurology_v1")

('checkpoints/flan_t5_neurology_v1\\tokenizer_config.json',
 'checkpoints/flan_t5_neurology_v1\\special_tokens_map.json',
 'checkpoints/flan_t5_neurology_v1\\tokenizer.json')

In [25]:
# Reload fine-tuned model + tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained("checkpoints/flan_t5_neurology_v1")
model = AutoModelForSeq2SeqLM.from_pretrained("checkpoints/flan_t5_neurology_v1")

In [26]:
# Tokenize the test dataset  
print("Tokenizing test dataset...")

test_tok = test_ds.map(
    preprocess_batch,           # reuse your earlier preprocessing function
    batched=True,
    remove_columns=test_ds.column_names
)

print("Test dataset tokenized successfully!")
print(test_tok)

Tokenizing test dataset...


Map:   0%|          | 0/169 [00:00<?, ? examples/s]C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 169/169 [00:00<00:00, 300.88 examples/s]

Test dataset tokenized successfully!
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 169
})


In [27]:
import torch
from tqdm import tqdm
import numpy as np
from transformers import BatchEncoding

# switch model to eval mode
model.eval()
model.config.num_beams = 1           # turn off beam search to save RAM
model.config.max_new_tokens = 192    # generation length cap

pred_texts, label_texts = [], []

print("\nGenerating predictions safely...")

# small batches to prevent memory overflow
for i in tqdm(range(0, len(test_tok), 4)):  # change 4 → 2 or 1 if still heavy
    batch = test_tok[i:i+4]
    # Convert batch to padded tensors
    batch_inputs = tok.pad(
        {k: batch[k] for k in ["input_ids", "attention_mask"]},
        padding=True,
        return_tensors="pt"
    )
    inputs = BatchEncoding(batch_inputs)
    with torch.no_grad():
        outputs = model.generate(**inputs)

    preds = tok.batch_decode(outputs, skip_special_tokens=True)
    labels = tok.batch_decode(batch["labels"], skip_special_tokens=True)

    pred_texts.extend(preds)
    label_texts.extend(labels)

print(f"Finished generation ({len(pred_texts)} predictions)")


Generating predictions safely...


100%|██████████| 43/43 [00:29<00:00,  1.46it/s]

Finished generation (169 predictions)


In [28]:
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
import numpy as np

# Compute BLEU
bleu = corpus_bleu(pred_texts, [label_texts]).score

# Compute ROUGE-L
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rougeL = np.mean([
    scorer.score(ref, hyp)['rougeL'].fmeasure
    for ref, hyp in zip(label_texts, pred_texts)
])

print(f"\nTest Set Metrics (manual, safe mode)")
print(f"BLEU     : {bleu:.2f}")
print(f"ROUGE-L  : {rougeL:.3f}")


Test Set Metrics (manual, safe mode)
BLEU     : 0.02
ROUGE-L  : 0.142


In [29]:
import sys
import subprocess

# Install bert-score directly into the current Python environment
subprocess.check_call([sys.executable, "-m", "pip", "install", "bert-score==0.3.13"])

0

In [30]:
from bert_score import score
import numpy as np

print("\nComputing BERTScore in memory-safe mode (small model)...")

# smaller multilingual models: good for laptops
MODEL_TYPE = "microsoft/deberta-base-mnli"  # or "bert-base-uncased"

batch_size = 20  # even smaller batch
P_all, R_all, F1_all = [], [], []

for i in range(0, len(pred_texts), batch_size):
    P, R, F1 = score(
        pred_texts[i:i+batch_size],
        label_texts[i:i+batch_size],
        lang="en",
        verbose=False,
        model_type=MODEL_TYPE,
        batch_size=batch_size,
        rescale_with_baseline=True
    )
    P_all.append(P.mean().item())
    R_all.append(R.mean().item())
    F1_all.append(F1.mean().item())

print(f"\n🧠 Semantic Metrics (lightweight BERTScore):")
print(f"Precision : {np.mean(P_all):.3f}")
print(f"Recall    : {np.mean(R_all):.3f}")
print(f"F1 (BERTScore): {np.mean(F1_all):.3f}")


Computing BERTScore in memory-safe mode (small model)...

🧠 Semantic Metrics (lightweight BERTScore):
Precision : 0.256
Recall    : -0.017
F1 (BERTScore): 0.104


In [31]:
# Reload the model from previous checkpoint
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

checkpoint_path = "checkpoints/flan_t5_neurology_v1"
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)

In [35]:
# TrainingArguments (v2 configuration)
from transformers import TrainingArguments

args_v2 = TrainingArguments(
    output_dir="checkpoints/flan_t5_neurology_v2",
    overwrite_output_dir=True,
    eval_strategy="epoch",   
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=6,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    logging_dir="./logs_v2",
    logging_steps=50,
    logging_first_step=True,
    report_to=None,  # disable W&B/TensorBoard completely
)

In [43]:
# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Define Trainer (metrics disabled during training)
trainer_v2 = Trainer(
    model=model,
    args=args_v2,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=None,  # disabled for speed & stability
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13140\2093758278.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_v2 = Trainer(


In [45]:
# Fine-tune (continue training)
train_output = trainer_v2.train()
print("\nRetraining complete!")
print(train_output)

Epoch,Training Loss,Validation Loss
1,2.520200,1.969581
2,2.171300,1.908625
3,2.277400,1.872742
4,2.214100,1.850741
5,2.212900,1.838386
6,2.183800,1.835114


C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(wa


Retraining complete!
TrainOutput(global_step=2028, training_loss=2.2681127489200947, metrics={'train_runtime': 6837.8039, 'train_samples_per_second': 1.184, 'train_steps_per_second': 0.297, 'total_flos': 69796438505472.0, 'train_loss': 2.2681127489200947, 'epoch': 6.0})


In [46]:
import torch
from tqdm import tqdm
from transformers import BatchEncoding
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer

model.eval()
model.config.num_beams = 1
model.config.max_new_tokens = 192

pred_texts, label_texts = [], []
print("\nGenerating predictions safely...")

for i in tqdm(range(0, len(val_tok), 4)):  # batch size = 4 (adjust if needed)
    batch = val_tok[i:i+4]
    batch_inputs = tokenizer.pad(
        {k: batch[k] for k in ["input_ids", "attention_mask"]},
        padding=True,
        return_tensors="pt"
    )
    inputs = BatchEncoding(batch_inputs)
    with torch.no_grad():
        outputs = model.generate(**inputs)
    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    labels = tokenizer.batch_decode(batch["labels"], skip_special_tokens=True)
    pred_texts.extend(preds)
    label_texts.extend(labels)

print(f"Generated {len(pred_texts)} samples.")

# Compute BLEU + ROUGE-L
bleu = corpus_bleu(pred_texts, [label_texts]).score
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rougeL = np.mean([scorer.score(ref, hyp)['rougeL'].fmeasure
                  for ref, hyp in zip(label_texts, pred_texts)])

print(f"\nEvaluation Metrics (safe mode)")
print(f"BLEU     : {bleu:.2f}")
print(f"ROUGE-L  : {rougeL:.3f}")


Generating predictions safely...


100%|██████████| 43/43 [00:29<00:00,  1.47it/s]


Generated 169 samples.

Evaluation Metrics (safe mode)
BLEU     : 0.03
ROUGE-L  : 0.185


In [47]:
# Save updated model
trainer_v2.save_model("checkpoints/flan_t5_neurology_v2")
tokenizer.save_pretrained("checkpoints/flan_t5_neurology_v2")

print("\nModel saved at checkpoints/flan_t5_neurology_v2")


Model saved at checkpoints/flan_t5_neurology_v2 ✅


In [48]:
# BERTScore Evaluation

print("\nComputing BERTScore safely (low-memory mode)...")

MODEL_TYPE = "bert-base-uncased"   # smaller and lighter model
BATCH_SIZE = 20                    # reduce if memory is still tight

P_all, R_all, F1_all = [], [], []

for i in range(0, len(pred_texts), BATCH_SIZE):
    batch_preds = pred_texts[i:i+BATCH_SIZE]
    batch_refs  = label_texts[i:i+BATCH_SIZE]
    
    P, R, F1 = score(
        batch_preds, batch_refs,
        lang="en",
        model_type=MODEL_TYPE,
        batch_size=BATCH_SIZE,
        verbose=False,
        rescale_with_baseline=True
    )
    
    P_all.append(P.mean().item())
    R_all.append(R.mean().item())
    F1_all.append(F1.mean().item())

print(f"\n🧠 Semantic Evaluation (BERTScore)")
print(f"Precision : {np.mean(P_all):.3f}")
print(f"Recall    : {np.mean(R_all):.3f}")
print(f"F1 Score  : {np.mean(F1_all):.3f}")


Computing BERTScore safely (low-memory mode)...


C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling bac


🧠 Semantic Evaluation (BERTScore)
Precision : 0.598
Recall    : 0.175
F1 Score  : 0.334


In [49]:
# Reload your previous fine-tuned model + tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained("checkpoints/flan_t5_neurology_v1")
tokenizer = AutoTokenizer.from_pretrained("checkpoints/flan_t5_neurology_v1")

In [50]:
# Training configuration
args_v3 = TrainingArguments(
    output_dir="checkpoints/flan_t5_neurology_v3",
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,            # continue up to 10 epochs
    weight_decay=0.01,
    save_total_limit=2,
    remove_unused_columns=False,
    logging_dir="./logs_v3",
    logging_strategy="steps",
    logging_steps=50,
    report_to=[],                   # disables W&B, TensorBoard
    seed=42,
)

In [51]:
# Trainer setup (no metrics during training)
trainer_v3 = Trainer(
    model=model,
    args=args_v3,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=None,
)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13140\2870686944.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_v3 = Trainer(


In [53]:
print("Resuming fine-tuning from previous checkpoint (flan_t5_neurology_v1)...")

trainer_v3.train()  # fresh training loop, continues from weights only

print("\nTraining complete! Model fine-tuned for 10 epochs safely.")

Resuming fine-tuning from previous checkpoint (flan_t5_neurology_v1)...


C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,2.632300,2.049038
2,2.241300,1.955462
3,2.327400,1.898494
4,2.239800,1.859940
5,2.218700,1.834741
6,2.161700,1.814968
7,2.148500,1.802751
8,2.210800,1.793698
9,2.118800,1.788664
10,2.063600,1.787094


C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(wa


Training complete! Model fine-tuned for 10 epochs safely.


In [54]:
# Save the Model
model.save_pretrained("checkpoints/flan_t5_neurology_v3")
tokenizer.save_pretrained("checkpoints/flan_t5_neurology_v3")

('checkpoints/flan_t5_neurology_v3\\tokenizer_config.json',
 'checkpoints/flan_t5_neurology_v3\\special_tokens_map.json',
 'checkpoints/flan_t5_neurology_v3\\tokenizer.json')

In [55]:
# Reload model and tokenizer
tok = AutoTokenizer.from_pretrained("checkpoints/flan_t5_neurology_v3")
model = AutoModelForSeq2SeqLM.from_pretrained("checkpoints/flan_t5_neurology_v3")
model.eval()

# Tokenize test dataset
print("Tokenizing test dataset...")
test_tok = test_ds.map(
    preprocess_batch,
    batched=True,
    remove_columns=test_ds.column_names
)
print("Test dataset ready.")

Tokenizing test dataset...


Map:   0%|          | 0/169 [00:00<?, ? examples/s]C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 169/169 [00:00<00:00, 1469.51 examples/s]

Test dataset ready.


In [56]:
# Generate predictions in safe batches
print("\nGenerating predictions safely...")
pred_texts, label_texts = [], []

model.config.num_beams = 1
model.config.max_new_tokens = 192

for i in tqdm(range(0, len(test_tok), 4)):  # adjust 4→2 or 1 if RAM spikes
    batch = test_tok[i:i+4]
    batch_inputs = tok.pad(
        {k: batch[k] for k in ["input_ids", "attention_mask"]},
        padding=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model.generate(**batch_inputs)
    preds = tok.batch_decode(outputs, skip_special_tokens=True)
    labels = tok.batch_decode(batch["labels"], skip_special_tokens=True)
    pred_texts.extend(preds)
    label_texts.extend(labels)

print(f"\nFinished generation ({len(pred_texts)} predictions)")


Generating predictions safely...


100%|██████████| 43/43 [00:27<00:00,  1.54it/s]


Finished generation (169 predictions)


In [58]:
from bert_score import score as bert_score

# Compute BLEU
bleu = corpus_bleu(pred_texts, [label_texts]).score

# Compute ROUGE-L
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rougeL = np.mean([
    scorer.score(ref, hyp)["rougeL"].fmeasure
    for ref, hyp in zip(label_texts, pred_texts)
])

# Compute BERTScore (lightweight)
print("\nComputing BERTScore (small batches)...")
MODEL_TYPE = "bert-base-uncased"   # safe small model for laptop
batch_size = 20
P_all, R_all, F1_all = [], [], []

for i in range(0, len(pred_texts), batch_size):
    P, R, F1 = bert_score(
        pred_texts[i:i+batch_size],
        label_texts[i:i+batch_size],
        lang="en",
        verbose=False,
        model_type=MODEL_TYPE,
        batch_size=batch_size,
        rescale_with_baseline=True
    )
    P_all.append(P.mean().item())
    R_all.append(R.mean().item())
    F1_all.append(F1.mean().item())

# Display results
print("\nFinal Evaluation Metrics")
print(f"BLEU      : {bleu:.2f}")
print(f"ROUGE-L   : {rougeL:.3f}")
print(f"BERTScore Precision : {np.mean(P_all):.3f}")
print(f"BERTScore Recall    : {np.mean(R_all):.3f}")
print(f"BERTScore F1        : {np.mean(F1_all):.3f}")


Computing BERTScore (small batches)...

Final Evaluation Metrics
BLEU      : 0.05
ROUGE-L   : 0.187
BERTScore Precision : 0.589
BERTScore Recall    : 0.187
BERTScore F1        : 0.339


In [59]:
# === Neurology Chatbot ===
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# load your final checkpoint
model_path = "checkpoints/flan_t5_neurology_v3"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
model.eval()

# generation settings (safe for CPU)
GEN_KWARGS = dict(
    max_new_tokens=192,
    num_beams=3,
    temperature=0.7,
    top_p=0.9,
    early_stopping=True,
    no_repeat_ngram_size=4
)

def generate_response(user_input):
    if not user_input.strip():
        return "Please enter a question related to neurology."
    prompt = f"Answer this neurology question clearly and concisely:\n\n{user_input}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(**inputs, **GEN_KWARGS)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# launch Gradio chat
demo = gr.ChatInterface(
    fn=lambda message, history: generate_response(message),
    title="Neurology Chatbot (FLAN-T5-Neurology-v3)",
    description=("A fine-tuned FLAN-T5 model answering neurology-related questions.\n"
                 "Note: Educational use only, not medical advice."),
    examples=[
        "What are the early signs of Parkinson's disease?",
        "Can multiple sclerosis cause vision problems?",
        "What is the difference between migraine with aura and without aura?",
        "How does epilepsy affect the brain?"
    ]
)
demo.launch()

C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
